# 🚀 머신러닝 실습 : 고객 구매 데이터로 성별 예측 모델링 (분류 문제)

* 주어진 데이터는 백화점 고객의 1년 간 구매 데이터입니다.
* 고객 3,500명에 대한 학습용 데이터(y.csv, X.csv)를 이용하여 성별예측 모형을 만들어보세요.
* 모델의 성능은 자유롭게 측정해봅니다!

## [실습 프로세스]
1. 데이터 불러오기  
2. 데이터 탐색
3. 데이터 전처리  
4. 학습/테스트 데이터 분리  
5. 모델 선택 및 학습  
6. 예측 및 평가  


<br/>

---

<br/>
<br/>

# 0. 라이브러리 불러오기

* 라이브러리를 가져와서 과정을 준비합니다

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

<br/>

---

<br/>
<br/>

# 1. 데이터 불러오기
* 데이터를 가져와서 과정을 준비합시다.
- 인코딩 방식은 'euc-kr' 을 활용하세요.
- 데이터 출처 : 한국데이터산업진흥원 빅데이터분석기사 실기 공개 예시 문항

- 독립 변수 데이터셋 : ./data/X.csv
- 종속 변수 데이터셋 : ./data/y.csv


데이터 파일을 불러옵니다. 보통 CSV 파일을 pandas로 읽어옵니다.

In [20]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: c:\githome\hipython_rep


In [186]:
X = pd.read_csv('./data1/X.csv', encoding='euc-kr')
y = pd.read_csv('./data1/y.csv', encoding='euc-kr')

<br/>

---

<br/>
<br/>

# 2. 데이터 탐색하기
* 데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.


데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.

In [4]:
X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [5]:
y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0



데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [6]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   object 
 5   주구매지점    3500 non-null   object 
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 273.6+ KB


In [7]:
X.describe()

,cust_id,총구매액,최대구매액,환불금액,내점일수,내점당구매건수,주말방문비율,구매주기
count,3500.000000,3.500000e+03,3.500000e+03,1.205000e+03,3500.000000,3500.000000,3500.000000,3500.000000
mean,1749.500000,9.191925e+07,1.966424e+07,2.407822e+07,19.253714,2.834963,0.307246,20.958286
std,1010.507298,1.635065e+08,3.199235e+07,4.746453e+07,27.174942,1.912368,0.289752,24.748682
min,0.000000,-5.242152e+07,-2.992000e+06,5.600000e+03,1.000000,1.000000,0.000000,0.000000
25%,874.750000,4.747050e+06,2.875000e+06,2.259000e+06,2.000000,1.666667,0.027291,4.000000
50%,1749.500000,2.822270e+07,9.837000e+06,7.392000e+06,8.000000,2.333333,0.256410,13.000000
75%,2624.250000,1.065079e+08,2.296250e+07,2.412000e+07,25.000000,3.375000,0.448980,28.000000
max,3499.000000,2.323180e+09,7.066290e+08,5.637530e+08,285.000000,22.083333,1.000000,166.000000


<br/>

---

<br/>
<br/>

# 3. 데이터 전처리
* 전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비


필요한 라이브러리를 불러옵니다.
- 인코딩 : LabelEncoder
- 데이터 표준화 : StandardScaler

In [12]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
le = LabelEncoder()

* 단순히 1부터의 숫자를 부여한 'cust_id'를 수치형 변수로 받아들이면, 결과가 왜곡될 수 있으니 컬럼을 제거합니다.

- 데이터에 결측치가 있는지 확인해보세요


- 결측치에 0으로 채워 넣어 모델 학습에 지장이 없도록 합니다.

In [187]:
X = X.drop('cust_id', axis=1)
y = y.drop('cust_id', axis=1)

In [188]:
X['환불금액'] = X['환불금액'].fillna(0)


문자형 범주 데이터를 숫자로 바꾸기 위한 인코딩을 수행합니다.

In [189]:
X['주구매상품'] = le.fit_transform(X['주구매상품'])
X['주구매지점'] = le.fit_transform(X['주구매지점'])
X.head()

,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,68282840,11264000,6860000.0,5,0,19,3.894737,0.527027,17
1,2136000,2136000,300000.0,21,19,2,1.500000,0.000000,1
2,3197000,1639000,0.0,6,1,2,2.000000,0.000000,1
3,16077620,4935000,0.0,5,2,18,2.444444,0.318182,16
4,29050000,24000000,0.0,15,8,2,1.500000,0.000000,85


각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.
- 평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [475]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X[['주구매상품', '주구매지점','주말방문비율','구매주기']], y, test_size=0.2, random_state=50)

In [476]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

<br/>

---

<br/>
<br/>

# 5-1. 모델링 - LogisticRegression

* 본격적으로 모델을 선언하고 학습시킵니다.


필요한 라이브러리를 불러옵니다.

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

모델을 선언하여 객체화시킵니다.

In [477]:
lr_clf = LogisticRegression()


모델을 학습 데이터에 맞춰 학습시킵니다.

In [478]:
lr_clf.fit(X_train_scaled, y_train)

LogisticRegression()

<br/>

---

<br/>
<br/>

# 6-1. 예측 성능 확인해보기 - LogisticRegression

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [479]:
lr_pred = lr_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [480]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(confusion_matrix(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

[[445   4]
 [246   5]]
              precision    recall  f1-score   support

           0       0.64      0.99      0.78       449
           1       0.56      0.02      0.04       251

    accuracy                           0.64       700
   macro avg       0.60      0.51      0.41       700
weighted avg       0.61      0.64      0.51       700




<br/>

---

<br/>
<br/>

# 5-2. 모델링 - DecisionTreeClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.


필요한 라이브러리를 불러옵니다.

In [334]:
from sklearn.tree import DecisionTreeClassifier

모델을 선언하여 객체화시킵니다.

In [481]:
dt_clf = DecisionTreeClassifier()

모델을 학습 데이터에 맞춰 학습시킵니다.

In [482]:
dt_clf.fit(X_train_scaled, y_train)

DecisionTreeClassifier()



<br/>
<br/>

# 6-2. 예측 성능 확인해보기 - DecisionTreeClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [483]:
dt_pred = dt_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [484]:
print(confusion_matrix(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

[[290 159]
 [156  95]]
              precision    recall  f1-score   support

           0       0.65      0.65      0.65       449
           1       0.37      0.38      0.38       251

    accuracy                           0.55       700
   macro avg       0.51      0.51      0.51       700
weighted avg       0.55      0.55      0.55       700



In [485]:
from sklearn.model_selection import GridSearchCV
params = {'max_depth' : [5,10,20,30], 'min_samples_split' : [5,10,20,30]}
grid_dtree = GridSearchCV(dt_clf, param_grid=params, cv=3, refit=True)
grid_dtree.fit(X_train_scaled, y_train)

GridSearchCV(cv=3, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': [5, 10, 20, 30],
                         'min_samples_split': [5, 10, 20, 30]})

In [486]:
grid_dtree.best_params_

{'max_depth': 5, 'min_samples_split': 5}

In [487]:
b_model = grid_dtree.best_estimator_
pred = b_model.predict(X_test_scaled)
accuracy_score(y_test, pred)

0.6342857142857142


<br/>

---

<br/>
<br/>

# 5-3. 모델링 - RandomForestClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.



필요한 라이브러리를 불러옵니다.

In [58]:
from sklearn.ensemble import RandomForestClassifier

모델을 선언하여 객체화시킵니다.

In [488]:
rf_clf = RandomForestClassifier(random_state=42, max_depth=8)

모델을 학습 데이터에 맞춰 학습시킵니다.

In [489]:
rf_clf.fit(X_train_scaled,y_train)

RandomForestClassifier(max_depth=8, random_state=42)



<br/>
<br/>

# 6-3. 예측 성능 확인해보기 - RandomForestClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [490]:
rf_pred = rf_clf.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [491]:
print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

[[383  66]
 [180  71]]
              precision    recall  f1-score   support

           0       0.68      0.85      0.76       449
           1       0.52      0.28      0.37       251

    accuracy                           0.65       700
   macro avg       0.60      0.57      0.56       700
weighted avg       0.62      0.65      0.62       700



In [492]:
from sklearn.model_selection import GridSearchCV
params = {'max_depth' : [4,5,6,7,8], 'min_samples_split' : [19,20,21]}
grid_dtree = GridSearchCV(rf_clf, param_grid=params, cv=3, refit=True)
grid_dtree.fit(X_train_scaled, y_train)

GridSearchCV(cv=3,
             estimator=RandomForestClassifier(max_depth=8, random_state=42),
             param_grid={'max_depth': [4, 5, 6, 7, 8],
                         'min_samples_split': [19, 20, 21]})

In [493]:
grid_dtree.best_params_

{'max_depth': 5, 'min_samples_split': 21}

In [494]:
b_model = grid_dtree.best_estimator_
pred = b_model.predict(X_test_scaled)
accuracy_score(y_test, pred)

0.6542857142857142


<br/>

---

<br/>
<br/>

# 5-4. 모델링 - XGBoost

* 본격적으로 모델을 선언하고 학습시킵니다.



필요한 라이브러리를 불러옵니다.

In [468]:
import xgboost
from xgboost import XGBClassifier

모델을 선언하여 객체화시킵니다.

In [495]:
xgb = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)

모델을 학습 데이터에 맞춰 학습시킵니다.

In [496]:
y_train_encoded = le.fit_transform(y_train)  # Series → 1D array
y_test_encoded = le.transform(y_test)

In [497]:
evals = [(X_test_scaled, y_test_encoded)]
xgb.fit(X_train_scaled, y_train_encoded, early_stopping_rounds=40, 
        eval_set=evals, verbose=True)

[0]	validation_0-logloss:0.68242
[1]	validation_0-logloss:0.67398
[2]	validation_0-logloss:0.66610
[3]	validation_0-logloss:0.65993
[4]	validation_0-logloss:0.65457
[5]	validation_0-logloss:0.64982
[6]	validation_0-logloss:0.64592
[7]	validation_0-logloss:0.64172
[8]	validation_0-logloss:0.63832
[9]	validation_0-logloss:0.63602
[10]	validation_0-logloss:0.63338
[11]	validation_0-logloss:0.63194
[12]	validation_0-logloss:0.63052
[13]	validation_0-logloss:0.62862
[14]	validation_0-logloss:0.62783
[15]	validation_0-logloss:0.62700
[16]	validation_0-logloss:0.62495
[17]	validation_0-logloss:0.62430
[18]	validation_0-logloss:0.62362
[19]	validation_0-logloss:0.62264
[20]	validation_0-logloss:0.62206
[21]	validation_0-logloss:0.62106
[22]	validation_0-logloss:0.62078
[23]	validation_0-logloss:0.61964
[24]	validation_0-logloss:0.61881
[25]	validation_0-logloss:0.61832
[26]	validation_0-logloss:0.61803
[27]	validation_0-logloss:0.61716
[28]	validation_0-logloss:0.61697
[29]	validation_0-loglos

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
              importance_type=None, interaction_constraints='',
              learning_rate=0.1, max_bin=256, max_cat_to_onehot=4,
              max_delta_step=0, max_depth=3, max_leaves=0, min_child_weight=1,
              missing=nan, monotone_constraints='()', n_estimators=400,
              n_jobs=0, num_parallel_tree=1, predictor='auto', random_state=0,
              reg_alpha=0, reg_lambda=1, ...)



<br/>
<br/>

# 6-4. 예측 성능 확인해보기 - XGBoost

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

In [498]:
xgb_pred = xgb.predict(X_test_scaled)

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [499]:
print(confusion_matrix(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

[[383  66]
 [173  78]]
              precision    recall  f1-score   support

           0       0.69      0.85      0.76       449
           1       0.54      0.31      0.39       251

    accuracy                           0.66       700
   macro avg       0.62      0.58      0.58       700
weighted avg       0.64      0.66      0.63       700



In [500]:
params = {'max_depth' : [1,2,3,4,5], 'min_samples_split' : [2,3,4,5]}
grid_dtree = GridSearchCV(xgb, param_grid=params, cv=3, refit=True)
grid_dtree.fit(X_train_scaled, y_train_encoded)
print(grid_dtree.best_params_)
b_model = grid_dtree.best_estimator_
pred = b_model.predict(X_test_scaled)
accuracy_score(y_test_encoded, pred)

[12:05:08] WARNING: D:\bld\xgboost-split_1666900898517\work\src\learner.cc:627: 
Parameters: { "min_samples_split" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[12:05:09] WARNING: D:\bld\xgboost-split_1666900898517\work\src\learner.cc:627: 
Parameters: { "min_samples_split" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[12:05:09] WARNING: D:\bld\xgboost-split_1666900898517\work\src\learner.cc:627: 
Parameters: { "min_samples_split" } might not be used.

  This could be a false alarm, with some parameters g

0.6614285714285715

<br/>

---


<br/>

## 7.  위 4가지 모델의 학습 & 예측 & 평가 결과를 확인하고 최고 성능을 내는 모델을 찾아봅시다!

- 어떤 모델이 가장 성능이 좋은가요 ?

XGBoost의 정확도가 66퍼로 가장 높았다